## What is new in models4 (vs models3)

- **Batched graph construction** via `build_batch` — graphs built slice by slice, so training starts after the first batch rather than waiting for the full dataset.
- **Evidence retry** via `retry_failed_evidence` — three fallback strategies (truncated headline → NER keywords → lead sentence) recover articles that initially return zero search results.
- **Warm-start fine-tuning** via `finetune_gat` — each subsequent batch fine-tunes from the previous model's weights at a lower LR.
- **Compounding credibility DB** — the source database is updated after every batch, enriching evidence scoring for later batches.
- **Visualised learning curve** — per-batch accuracy / F1 / AUC on a fixed global validation pool.

---

# models4.ipynb — Batched Build & Incremental Training

## Overview

This notebook builds a **Graph Attention Network (GAT)** for fake-news detection on the
[ISOT dataset](https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets).
It is the third iteration in this series; see `models2.ipynb` for the previous version.

Each news article is converted into a **Structured Argumentation Graph**: a typed graph
where nodes represent sentences (labelled by rhetorical role) and edges encode semantic
similarity and logical relations (entailment / contradiction / neutral) both within the
article and across externally retrieved articles covering the same event.

---

## Core redesign from models2

### What was wrong
`models2` searched using entity queries extracted from the article body. This produced
irrelevant results (emojipedia, dndbeyond, tacomaworld) and left **262 / 500 graphs with
zero evidence nodes** — graphs that all look structurally identical and give the model
nothing to learn from.

### What changes

**1. Headline-based search** *(O(1) per article, not O(n_claims))*  
Headlines are written to be precise and findable. Searching the headline directly
retrieves articles covering the *same event* from different publishers — exactly the
cross-source comparison we want.

**2. Sentence role classification** *(zero-shot NLI)*  
Each sentence is labelled as one of: `claim`, `evidence`, `analysis`, `background`.
Misinformation manipulates the *analysis* layer while keeping evidence plausible — so
this distinction is the key signal that was missing.

**3. Cross-source analysis comparison**  
The target article's analysis sentences are compared via NLI to analysis sentences from
retrieved articles covering the same topic. Analysis entailed by multiple independent
sources is credible; analysis that contradicts them is a fake signal.

**4. Typed graph structure**  
Nodes carry role labels. Edges connect: intra-article (sentence↔sentence) and
cross-source (target analysis ↔ retrieved analysis). Node features include role type and
NLI relation counts broken down by role.

---

**Setup:** place `Fake.csv` and `True.csv` from the ISOT dataset in a `data/` folder.

In [1]:
import os, json, hashlib, math, time, sqlite3
from contextlib import contextmanager
from urllib.parse import urlparse
from datetime import datetime, timezone

import nltk
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    nltk.download('punkt', quiet=True)

import spacy
nlp = spacy.load('en_core_web_sm')

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
import numpy as np
import networkx as nx
from nltk.tokenize import sent_tokenize

ENCODER   = SentenceTransformer('all-MiniLM-L6-v2')
NLI_MODEL = CrossEncoder('cross-encoder/nli-deberta-v3-small')

# NLI label indices for nli-deberta-v3-small
NLI_CONTRADICTION = 0
NLI_ENTAILMENT    = 1
NLI_NEUTRAL       = 2

# Edge type constants
EDGE_NEUTRAL       = 0
EDGE_ENTAILMENT    = 1
EDGE_CONTRADICTION = 2

# Sentence role constants
ROLE_CLAIM      = 0
ROLE_EVIDENCE   = 1
ROLE_ANALYSIS   = 2
ROLE_BACKGROUND = 3
ROLE_NAMES      = ['claim', 'evidence', 'analysis', 'background']

# Node feature dimension — expanded to include role encoding
FEATURE_DIM = 20

print('Setup complete.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\bhada\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhada\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete.


## Step 1: Load ISOT Dataset

The ISOT Fake News Dataset contains ~23 000 fake articles (from unreliable sources
flagged by fact-checking organisations) and ~21 000 real articles (from Reuters.com).
We shuffle, binary-encode the label, and strip the Reuters dateline from real articles
to prevent the model from learning a trivial formatting heuristic.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

fake = pd.read_csv('data/Fake.csv')
real = pd.read_csv('data/True.csv')
fake['label'] = 'fake'
real['label'] = 'real'

df = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)
df['label_binary'] = (df['label'] == 'fake').astype(int)

# Strip Reuters dateline from real articles to prevent trivial leakage
df['text'] = df['text'].str.replace(
    r'^[A-Z\s,]+\([^)]+\)\s*-\s*', '', regex=True
).str.strip()

print(f'Total: {len(df)} | Balance: {df["label_binary"].value_counts().to_dict()}')
df[['title', 'text', 'label']].head(2)

Total: 44898 | Balance: {1: 23481, 0: 21417}


,title,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",fake
1,Trump drops Steve Bannon from National Securit...,U.S. President Donald Trump removed his chief ...,real


## Step 2: Sentence Role Classification

Each sentence is classified as **claim**, **evidence**, **analysis**, or **background**
using zero-shot NLI. Rather than fine-tuning a dedicated classifier, we run the NLI
model against four defining hypotheses and pick the highest-scoring one.

This is the key structural addition over `models2`. Misinformation rarely fabricates raw
events — it manipulates the *analysis* layer: presenting selective evidence, drawing
unsupported conclusions, or framing neutral facts with loaded interpretation. Making this
distinction explicit in the graph gives the model the right signal to learn from.

**Why not fine-tune a classifier?** Zero-shot NLI generalises across topics without any
labelled role data. A fine-tuned classifier would need a role-labelled news corpus that
doesn't readily exist and would risk overfitting to surface patterns of specific
publications.

In [3]:
# Defining hypotheses for each role — written to activate the NLI model's
# entailment signal when the premise (sentence) matches the role description.
ROLE_HYPOTHESES = [
    "This sentence makes a specific factual assertion or claim.",      # ROLE_CLAIM
    "This sentence presents data, quotes, or cited supporting evidence.", # ROLE_EVIDENCE
    "This sentence provides interpretation, opinion, or analysis.",    # ROLE_ANALYSIS
    "This sentence provides background context or general information.", # ROLE_BACKGROUND
]


def classify_sentence_roles(sentences: list[str]) -> list[int]:
    """
    Classify each sentence into one of 4 roles using zero-shot NLI.

    For each sentence we run NLI against all 4 hypotheses in a single
    batch call. The hypothesis with the highest entailment score wins.

    Batching all sentences × all hypotheses in one predict() call is
    much faster than calling predict() once per sentence.
    """
    if not sentences:
        return []

    # Build all (sentence, hypothesis) pairs
    pairs = [
        (sent, hyp)
        for sent in sentences
        for hyp in ROLE_HYPOTHESES
    ]

    # Batch NLI inference
    logits    = NLI_MODEL.predict(pairs)                          # (n_sent * 4, 3)
    probs     = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    ent_probs = probs[:, NLI_ENTAILMENT]                          # entailment probability only

    # Reshape to (n_sentences, 4) and argmax per sentence
    ent_matrix = ent_probs.reshape(len(sentences), len(ROLE_HYPOTHESES))
    roles      = np.argmax(ent_matrix, axis=1).tolist()
    return roles


# Sanity check
test_sents = [
    "President Trump signed an executive order on Friday.",
    "According to documents obtained by the Times, the memo was dated March 3.",
    "This represents a fundamental shift in American foreign policy.",
    "The conflict between the two nations began in 1948."
]
roles = classify_sentence_roles(test_sents)
for s, r in zip(test_sents, roles):
    print(f'[{ROLE_NAMES[r]:<12}] {s}')

[claim       ] President Trump signed an executive order on Friday.
[background  ] According to documents obtained by the Times, the memo was dated March 3.
[claim       ] This represents a fundamental shift in American foreign policy.
[analysis    ] The conflict between the two nations began in 1948.


## Step 3: Headline Search

We search using the **article headline** instead of entity queries from the body text.

**Why headlines work better:**
- Headlines are purpose-written to be precise and findable — they are the journalist's best distillation of the article's core claim
- A headline search retrieves articles covering the *same event* from different publishers, which is exactly the cross-source comparison we want
- Entity queries from body text match individual entities that may appear in completely unrelated contexts (hence emojipedia and dndbeyond appearing as evidence)

This reduces from O(n_claims) search calls to O(1) per article.

In [4]:
from duckduckgo_search import DDGS

CACHE_DIR = 'search_cache_v3'  # separate cache from models2 to avoid stale results


def _cache_path(query: str) -> str:
    os.makedirs(CACHE_DIR, exist_ok=True)
    return os.path.join(CACHE_DIR, hashlib.md5(query.encode()).hexdigest() + '.json')


def ddg_search(query: str, num_results: int = 10) -> list[dict]:
    cache_file = _cache_path(query)
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=num_results):
            results.append({
                'link':    r.get('href', ''),
                'title':   r.get('title', ''),
                'snippet': r.get('body', '')
            })
    with open(cache_file, 'w') as f:
        json.dump(results, f)
    time.sleep(0.5)
    return results


def clean_headline(title: str) -> str:
    """
    Strip publication suffixes from headlines before searching.
    e.g. 'Biden signs bill | Reuters' → 'Biden signs bill'
    These suffixes bias results back toward the same publisher.
    """
    for sep in [' | ', ' - ', ' – ', ' — ']:
        if sep in title:
            title = title.split(sep)[0]
    return title.strip()


def collect_evidence_by_headline(title: str, k: int = 10) -> list[dict]:
    """
    Search using the article headline to find coverage of the same event
    from different publishers. Returns raw search result dicts.
    """
    query = clean_headline(title)
    if not query or len(query) < 10:
        return []
    try:
        return ddg_search(query, num_results=k)
    except Exception as e:
        print(f'  Search failed for "{query[:60]}": {e}')
        return []

## Step 4: Source Credibility Database

We maintain a lightweight per-domain **Bayesian credibility score** backed by SQLite.
Each domain starts with a Beta(2, 2) prior (score = 0.5 — no information).

- **Model updates**: when the GAT classifies an article with high confidence, every
  domain that contributed an evidence node receives a fractional update weighted by
  the document's relevance score and the model's confidence.
- **User updates**: explicit human labels can be injected with weight 1.0 (vs 0.3 for
  model updates) to let ground-truth feedback dominate.

The credibility score is blended with the DBSCAN consensus score when ranking retrieved
documents, so high-credibility sources get proportionally more edge weight in the graph.

In [5]:
DB_PATH      = 'source_credibility_v3.db'
MODEL_WEIGHT = 0.3
USER_WEIGHT  = 1.0


@contextmanager
def get_db():
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def init_db():
    with get_db() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS sources (
                domain        TEXT PRIMARY KEY,
                alpha         REAL DEFAULT 2.0,
                beta          REAL DEFAULT 2.0,
                model_updates INTEGER DEFAULT 0,
                user_updates  INTEGER DEFAULT 0,
                last_updated  TEXT
            )''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS credibility_log (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                domain      TEXT,
                signal_type TEXT,
                signal      TEXT,
                confidence  REAL,
                timestamp   TEXT
            )''')

init_db()


def extract_domain(url: str) -> str:
    return urlparse(url).netloc.replace('www.', '')


def get_credibility(domain: str) -> float:
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta FROM sources WHERE domain = ?', (domain,)
        ).fetchone()
    return (row[0] / (row[0] + row[1])) if row else 0.5


def update_credibility(domain: str, signal: str, signal_type: str, confidence: float = 1.0):
    assert signal in ('real', 'fake') and signal_type in ('model', 'user')
    weight = (MODEL_WEIGHT if signal_type == 'model' else USER_WEIGHT) * confidence
    now    = datetime.now(timezone.utc).isoformat()  # fixed deprecation warning
    with get_db() as conn:
        conn.execute(
            'INSERT OR IGNORE INTO sources (domain,alpha,beta,last_updated) VALUES(?,2.0,2.0,?)',
            (domain, now)
        )
        col        = 'alpha' if signal == 'real' else 'beta'
        update_col = 'model_updates' if signal_type == 'model' else 'user_updates'
        conn.execute(
            f'UPDATE sources SET {col}={col}+?,{update_col}={update_col}+1,last_updated=? WHERE domain=?',
            (weight, now, domain)
        )
        conn.execute(
            'INSERT INTO credibility_log(domain,signal_type,signal,confidence,timestamp) VALUES(?,?,?,?,?)',
            (domain, signal_type, signal, confidence, now)
        )


def bulk_update_from_prediction(scored_docs, prediction: float,
                                 model_confidence: float,
                                 confidence_threshold: float = 0.1) -> int:
    if model_confidence < confidence_threshold:
        return 0
    signal = 'fake' if prediction > 0.5 else 'real'
    n = 0
    for doc, doc_score in scored_docs:
        url = doc.get('link', '')
        if url:
            update_credibility(extract_domain(url), signal, 'model',
                               confidence=model_confidence * doc_score)
            n += 1
    return n


print('DB ready at', DB_PATH)

DB ready at source_credibility_v3.db


## Step 5: Structured Argumentation Graph

The graph now has two kinds of nodes and three kinds of edges:

**Nodes:**
- `input_*` — sentences from the target article, labeled by role (claim/evidence/analysis/background)
- `ext_*` — sentences from retrieved articles, labeled by role

**Edges:**
- **Intra-article** (input↔input): cosine similarity > threshold, then NLI-typed
- **Cross-source analysis** (input_analysis↔ext_analysis): NLI comparison between target article's analysis and retrieved articles' analysis — this is the core new signal
- **Cross-source evidence** (input_claim↔ext_evidence): NLI comparison between target claims and retrieved evidence

Cross-source edges are only drawn between matching role pairs to avoid noise. Comparing a background sentence to an analysis sentence from another source produces meaningless NLI scores.

In [6]:
def extract_and_embed(text: str) -> tuple[list[str], np.ndarray]:
    """Tokenize text into sentences and embed all at once."""
    sentences = [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 10]
    if not sentences:
        return [], np.array([])
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return sentences, embeddings


def nli_batch(pairs: list[tuple[str, str]]) -> np.ndarray:
    """Run NLI on a list of (premise, hypothesis) pairs. Returns (n, 3) prob array."""
    if not pairs:
        return np.array([])
    logits = NLI_MODEL.predict(pairs)
    return np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)


def edge_type_from_probs(probs_row: np.ndarray) -> tuple[int, float]:
    label = int(np.argmax(probs_row))
    etype = EDGE_CONTRADICTION if label == NLI_CONTRADICTION else \
            EDGE_ENTAILMENT    if label == NLI_ENTAILMENT    else EDGE_NEUTRAL
    return etype, float(probs_row[label])


def build_article_graph(text: str,
                         sim_threshold: float = 0.60,
                         use_nli: bool = True) -> tuple[nx.Graph, list[str], list[int]]:
    """
    Build the base knowledge graph from the article text.
    Returns the graph, the list of sentences, and their role labels.
    """
    sentences, embeddings = extract_and_embed(text)
    G = nx.Graph()
    if not sentences:
        return G, [], []

    roles = classify_sentence_roles(sentences)

    for i, (sent, emb, role) in enumerate(zip(sentences, embeddings, roles)):
        G.add_node(i, text=sent, embedding=emb, source='input',
                   role=role, weight=1.0)

    sim_matrix = cosine_similarity(embeddings)
    candidate_pairs = [
        (i, j) for i in range(len(sentences))
        for j in range(i + 1, len(sentences))
        if sim_matrix[i, j] > sim_threshold
    ]

    if use_nli and candidate_pairs:
        pairs_text = [(sentences[i], sentences[j]) for i, j in candidate_pairs]
        probs_all  = nli_batch(pairs_text)
        for (i, j), probs in zip(candidate_pairs, probs_all):
            etype, conf = edge_type_from_probs(probs)
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=etype, nli_confidence=conf,
                       edge_source='intra')
    else:
        for i, j in candidate_pairs:
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=EDGE_NEUTRAL, nli_confidence=0.5,
                       edge_source='intra')

    return G, sentences, roles


def augment_with_cross_source(G: nx.Graph,
                               article_sentences: list[str],
                               article_roles: list[int],
                               retrieved_docs: list[dict],
                               sim_threshold: float = 0.55,
                               score_threshold: float = 0.2,
                               use_nli: bool = True) -> tuple[nx.Graph, list[tuple[dict, float]]]:
    """
    Augment the graph with sentences from retrieved articles.

    Cross-source edges are only drawn between semantically similar sentences
    of *matching roles*:
      - input analysis ↔ ext analysis  (is our interpretation corroborated?)
      - input claim    ↔ ext evidence  (is our assertion backed externally?)
      - input claim    ↔ ext claim     (do other sources assert the same thing?)

    Comparing background to analysis across sources produces noise.

    Returns the augmented graph and a list of (doc, score) for the DB update.
    """
    if not retrieved_docs:
        return G, []

    # Score documents by DBSCAN consensus + credibility
    scored_docs = score_documents(retrieved_docs)

    # Get input node info for cross-source comparison
    input_nodes = [(nid, data) for nid, data in G.nodes(data=True)
                   if data.get('source') == 'input']

    # Index article sentences by role for efficient lookup
    article_embs = np.array([data['embedding'] for _, data in input_nodes])

    next_id = max(G.nodes()) + 1 if G.nodes() else 0
    new_nodes, new_edges = [], []

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold or not doc.get('snippet'):
            continue

        ext_sents, ext_embs = extract_and_embed(doc['snippet'])
        if not ext_sents:
            continue

        ext_roles = classify_sentence_roles(ext_sents)

        # Compute similarity between all external and all input sentences at once
        sim_matrix = cosine_similarity(ext_embs, article_embs)  # (n_ext, n_input)

        # Collect candidate cross-source pairs
        cross_pairs_text  = []
        cross_pairs_index = []  # (ext_idx, input_node_idx)

        for ext_i, (ext_sent, ext_role) in enumerate(zip(ext_sents, ext_roles)):
            for inp_j, (inp_nid, inp_data) in enumerate(input_nodes):
                if sim_matrix[ext_i, inp_j] < sim_threshold:
                    continue
                inp_role = inp_data['role']

                # Only compare matching or complementary role pairs
                valid = (
                    (ext_role == ROLE_ANALYSIS   and inp_role == ROLE_ANALYSIS)  or
                    (ext_role == ROLE_EVIDENCE   and inp_role == ROLE_CLAIM)     or
                    (ext_role == ROLE_CLAIM      and inp_role == ROLE_CLAIM)     or
                    (ext_role == ROLE_CLAIM      and inp_role == ROLE_ANALYSIS)
                )
                if valid:
                    cross_pairs_text.append((ext_sent, inp_data['text']))
                    cross_pairs_index.append((ext_i, inp_j, inp_nid,
                                              float(sim_matrix[ext_i, inp_j]),
                                              ext_role))

        # Batch NLI on all cross-source pairs for this document
        if use_nli and cross_pairs_text:
            cross_probs = nli_batch(cross_pairs_text)
        else:
            cross_probs = np.full((len(cross_pairs_text), 3), 1/3)

        # Add external sentences as nodes and draw edges
        ext_node_ids = {}
        for pi, (ext_i, inp_j, inp_nid, sim, ext_role) in enumerate(cross_pairs_index):
            # Add external node if not already added for this doc
            if ext_i not in ext_node_ids:
                nid = next_id
                next_id += 1
                ext_node_ids[ext_i] = nid
                new_nodes.append((
                    nid, ext_sents[ext_i], ext_embs[ext_i], doc_score, ext_role
                ))

            ext_nid = ext_node_ids[ext_i]
            etype, conf = edge_type_from_probs(cross_probs[pi])
            new_edges.append((
                ext_nid, inp_nid, sim, doc_score, etype, conf, 'cross_source'
            ))

    # Commit all changes to G
    for nid, text, emb, score, role in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence',
                   role=role, weight=score)
    for nid, inp_nid, sim, score, etype, conf, esrc in new_edges:
        G.add_edge(nid, inp_nid, similarity=sim, evidence_weight=score,
                   edge_type=etype, nli_confidence=conf, edge_source=esrc)

    return G, scored_docs


def score_documents(documents: list[dict],
                    credibility_alpha: float = 0.6,
                    eps: float = 0.3,
                    min_samples: int = 2) -> list[tuple[dict, float]]:
    """DBSCAN consensus clustering + domain credibility blending."""
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []
    embeddings  = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    sim_matrix  = cosine_similarity(embeddings)
    dist_matrix = np.clip(1.0 - sim_matrix, 0, 2).astype(np.float64)
    labels      = DBSCAN(eps=eps, min_samples=min_samples,
                         metric='precomputed').fit_predict(dist_matrix)
    unique      = [l for l in set(labels) if l != -1]
    cons_label  = max(unique, key=lambda l: (labels == l).sum()) if unique else None

    scored = []
    for i, doc in enumerate(documents):
        if cons_label is not None:
            if labels[i] == cons_label:
                members = np.where(labels == cons_label)[0]
                cs      = float(sim_matrix[i, members].mean())
            elif labels[i] == -1:
                cs = 0.1
            else:
                cs = float(sim_matrix[i].mean()) * 0.5
        else:
            cs = float((sim_matrix[i].sum() - 1.0) / max(len(documents) - 1, 1))

        domain   = extract_domain(doc.get('link', ''))
        combined = credibility_alpha * cs + (1 - credibility_alpha) * get_credibility(domain)
        scored.append((doc, float(combined)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

## Step 6: Node Features (20-dimensional)

Each node in the graph is described by a 20-dimensional hand-crafted feature vector.
Features 0–13 were present in `models2`; features 14–19 are new in this version.

| Index | Feature | Description |
|-------|---------|-------------|
| 0 | `n_evidence_nbrs` | Number of external (evidence) neighbours |
| 1 | `n_input_nbrs` | Number of intra-article neighbours |
| 2 | `mean_ev_weight` | Mean document score of evidence neighbours |
| 3 | `max_ev_weight` | Max document score of evidence neighbours |
| 4 | `mean_sim` | Mean edge cosine similarity |
| 5 | `ev_ratio` | Fraction of neighbours that are external |
| 6 | `is_evidence` | 1 if this node is from an external source |
| 7 | `node_weight` | Document credibility / relevance score |
| 8 | `n_entailing` | Count of entailment edges |
| 9 | `n_contradicting` | Count of contradiction edges |
| 10 | `ent_wsum` | Weighted entailment sum (weight × confidence) |
| 11 | `cont_wsum` | Weighted contradiction sum |
| 12 | `ent_ratio` | Entailment fraction of evidence neighbours |
| 13 | `cont_ratio` | Contradiction fraction of evidence neighbours |
| 14–17 | `role_onehot` | One-hot role: claim / evidence / analysis / background |
| 18 | `cross_ent_w` | Cross-source entailment weight (corroboration signal) |
| 19 | `cross_cont_w` | Cross-source contradiction weight (dispute signal) |

In [7]:
import torch
from torch_geometric.data import Data


def compute_node_features(G: nx.Graph) -> np.ndarray:
    feature_list = []
    for node_id, data in G.nodes(data=True):
        neighbors     = list(G.neighbors(node_id))
        ev_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'evidence']
        in_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'input']
        ev_weights    = [G.edges[node_id, n].get('evidence_weight', 0.0) for n in ev_nbrs]
        all_sims      = [G.edges[node_id, n].get('similarity', 0.0) for n in neighbors]

        # NLI breakdown — all edges
        entailing     = [(n, G.edges[node_id, n]) for n in ev_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_ENTAILMENT]
        contradicting = [(n, G.edges[node_id, n]) for n in ev_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_CONTRADICTION]

        ent_wsum  = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                        for _, e in entailing)
        cont_wsum = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                        for _, e in contradicting)
        n_ev = max(len(ev_nbrs), 1)

        # Cross-source specific breakdown
        cross_nbrs = [n for n in ev_nbrs
                      if G.edges[node_id, n].get('edge_source') == 'cross_source']
        cross_ent  = [(n, G.edges[node_id, n]) for n in cross_nbrs
                      if G.edges[node_id, n].get('edge_type') == EDGE_ENTAILMENT]
        cross_cont = [(n, G.edges[node_id, n]) for n in cross_nbrs
                      if G.edges[node_id, n].get('edge_type') == EDGE_CONTRADICTION]

        cross_ent_w  = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                           for _, e in cross_ent)
        cross_cont_w = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                           for _, e in cross_cont)

        # Role one-hot
        role    = data.get('role', ROLE_BACKGROUND)
        role_oh = [1.0 if role == r else 0.0 for r in range(4)]

        feature_list.append([
            float(len(ev_nbrs)),                                       # 0
            float(len(in_nbrs)),                                       # 1
            float(np.mean(ev_weights) if ev_weights else 0.0),         # 2
            float(np.max(ev_weights)  if ev_weights else 0.0),         # 3
            float(np.mean(all_sims)   if all_sims   else 0.0),         # 4
            float(len(ev_nbrs) / max(len(neighbors), 1)),              # 5
            1.0 if data.get('source') == 'evidence' else 0.0,          # 6
            float(data.get('weight', 1.0)),                            # 7
            float(len(entailing)),                                     # 8
            float(len(contradicting)),                                 # 9
            ent_wsum,                                                  # 10
            cont_wsum,                                                 # 11
            float(len(entailing)    / n_ev),                           # 12
            float(len(contradicting) / n_ev),                          # 13
            *role_oh,                                                  # 14-17
            cross_ent_w,                                               # 18
            cross_cont_w,                                              # 19
        ])

    arr     = np.array(feature_list, dtype=np.float32)
    col_max = arr.max(axis=0)
    col_max[col_max == 0] = 1
    return arr / col_max


def graph_to_pyg(G: nx.Graph, label: int | None = None) -> Data:
    node_ids = list(G.nodes())
    id_map   = {nid: i for i, nid in enumerate(node_ids)}
    x        = torch.tensor(compute_node_features(G), dtype=torch.float)

    if G.edges():
        edges      = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edges, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_feats = []
        for u, v in G.edges():
            ed  = G.edges[u, v]
            et  = ed.get('edge_type', EDGE_NEUTRAL)
            esrc = 1.0 if ed.get('edge_source') == 'cross_source' else 0.0
            oh  = [1.0 if et == k else 0.0 for k in range(3)]
            edge_feats.append(oh + [
                ed.get('similarity', 0.0),
                ed.get('nli_confidence', 0.5),
                ed.get('evidence_weight', 0.0),
                esrc  # whether this is a cross-source edge
            ])
        ef_tensor = torch.tensor(edge_feats * 2, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        ef_tensor  = torch.zeros((0, 7), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=ef_tensor)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data

## Step 7: Graph Attention Network (GAT)

The classifier is a 4-layer **Graph Attention Network** with the following design choices:

- **Input projection**: a linear layer maps the 20-dim node features into the hidden
  space before any message passing. This decouples feature scale from hidden dimension.
- **4 × GATConv layers** with multi-head attention (4 heads) and skip connections.
  The first three layers use `concat=True` (outputs are concatenated across heads);
  the final layer uses `concat=False` (outputs are averaged) to control the growth of
  the hidden dimension.
- **Jumping Knowledge (JK) aggregation**: representations from all four layers are
  concatenated (`xjk`), so the readout can draw on local *and* long-range structure
  simultaneously.
- **Global pooling**: both `global_mean_pool` and `global_max_pool` are applied to
  `xjk` and concatenated. Mean captures average node behaviour; max captures the most
  extreme signal anywhere in the graph.
- **MLP classifier**: a 3-layer MLP with BatchNorm, ReLU, and Dropout maps the pooled
  graph representation to a single logit (binary cross-entropy loss).
- **Training**: Adam with cosine annealing LR schedule, gradient clipping, and early
  stopping on validation loss.

In [8]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader


class FakeNewsGAT(torch.nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden_dim=64, n_heads=4, dropout=0.4):
        super().__init__()
        self.dropout    = dropout
        self.input_proj = torch.nn.Linear(input_dim, hidden_dim)

        self.conv1 = GATConv(hidden_dim,     hidden_dim,     heads=n_heads, concat=True,  dropout=dropout)
        self.lin1  = torch.nn.Linear(hidden_dim * n_heads, hidden_dim)
        self.bn1   = torch.nn.BatchNorm1d(hidden_dim)

        self.conv2 = GATConv(hidden_dim,     hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin2  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn2   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv3 = GATConv(hidden_dim * 2, hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin3  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn3   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv4 = GATConv(hidden_dim * 2, hidden_dim,     heads=n_heads, concat=False, dropout=dropout)
        self.bn4   = torch.nn.BatchNorm1d(hidden_dim)

        pool_dim = (hidden_dim + hidden_dim*2 + hidden_dim*2 + hidden_dim) * 2
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(pool_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Linear(256, 64),  torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, batch):
        x    = F.relu(self.input_proj(x))
        x    = F.dropout(x, p=self.dropout, training=self.training)
        x1   = self.bn1(F.relu(self.lin1(self.conv1(x,  edge_index)))) + x   # residual: hidden → hidden
        x2   = self.bn2(F.relu(self.lin2(self.conv2(x1, edge_index))))        # no residual: dim doubles
        x3   = self.bn3(F.relu(self.lin3(self.conv3(x2, edge_index)))) + x2  # residual: hidden*2 → hidden*2
        x4   = self.bn4(F.relu(self.conv4(x3, edge_index))) + x3[:, :x1.shape[1]]  # residual: hidden*2 → hidden
        xjk  = torch.cat([x1, x2, x3, x4], dim=1)
        xp   = torch.cat([global_mean_pool(xjk, batch), global_max_pool(xjk, batch)], dim=1)
        return self.mlp(xp)


def train_gat(train_data, val_data, epochs=150, lr=5e-4, patience=20):
    model     = FakeNewsGAT()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.BCEWithLogitsLoss()
    t_loader  = DataLoader(train_data, batch_size=16, shuffle=True)
    v_loader  = DataLoader(val_data,   batch_size=16)

    best_val, no_improve, best_state = float('inf'), 0, None
    for epoch in range(epochs):
        model.train()
        t_loss = 0
        for b in t_loader:
            optimizer.zero_grad()
            out  = model(b.x, b.edge_index, b.batch).squeeze()
            loss = criterion(out, b.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        scheduler.step()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for b in v_loader:
                v_loss += criterion(
                    model(b.x, b.edge_index, b.batch).squeeze(), b.y.squeeze()
                ).item()

        avg_t, avg_v = t_loss / len(t_loader), v_loss / len(v_loader)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}  train={avg_t:.4f}  val={avg_v:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.2e}')

        if avg_v < best_val:
            best_val, no_improve = avg_v, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, dataset):
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in DataLoader(dataset, batch_size=16):
            probs = torch.sigmoid(model(b.x, b.edge_index, b.batch).squeeze())
            preds.extend(probs.tolist())
            labels.extend(b.y.squeeze().tolist())
    binary = [1 if p > 0.5 else 0 for p in preds]
    return {
        'accuracy': accuracy_score(labels, binary),
        'f1':       f1_score(labels, binary, zero_division=0),
        'auc':      roc_auc_score(labels, preds)
    }, preds


def save_model(model, path):
    torch.save(model.state_dict(), path)
    print(f'Saved to {path}')

def load_model(path):
    m = FakeNewsGAT()
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    return m

## Step 8: Build Dataset

For each article we:
1. Tokenise the body into sentences and classify their roles (NLI pass).
2. Search DuckDuckGo with the article headline and retrieve up to 10 results.
3. Score the retrieved documents by DBSCAN consensus + domain credibility.
4. Augment the intra-article graph with cross-source edges (role-matched NLI pairs).
5. Compute the 20-dim node feature matrix and convert to a PyG `Data` object.

> **Note:** this build is slower per article than `models2` because role classification
> adds one full NLI pass per article. However it produces far fewer empty graphs since
> headline search returns topically relevant results for almost every article.

In [9]:
from tqdm import tqdm


def build_dataset(df, text_col='text', title_col='title',
                  label_col='label_binary', sample=None, use_nli=True):
    if sample:
        df = df.groupby(label_col, group_keys=False).apply(
            lambda g: g.sample(sample // 2, random_state=42)
        ).reset_index(drop=True)

    pyg_data, all_scored = [], []
    n_aug, n_fall = 0, 0
    role_counts = {r: 0 for r in ROLE_NAMES}

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        text  = row[text_col]
        title = row.get(title_col, '')
        label = int(row[label_col])

        # Build base article graph with role-labeled nodes
        G, article_sents, article_roles = build_article_graph(text, use_nli=use_nli)
        if not article_sents:
            all_scored.append([])
            continue

        # Track role distribution for diagnostics
        for r in article_roles:
            role_counts[ROLE_NAMES[r]] += 1

        # Headline search + cross-source augmentation
        scored_docs = []
        try:
            docs = collect_evidence_by_headline(title, k=10)
            if docs:
                G, scored_docs = augment_with_cross_source(
                    G, article_sents, article_roles, docs, use_nli=use_nli
                )
                n_aug += 1
            else:
                n_fall += 1
        except Exception as e:
            print(f'  Augmentation failed: {e}')
            n_fall += 1

        pyg_data.append(graph_to_pyg(G, label=label))
        all_scored.append(scored_docs)

    total_roles = sum(role_counts.values())
    print(f'\nBuilt {len(pyg_data)} | Augmented: {n_aug} | Fallback: {n_fall}')
    print('Role distribution:', {k: f'{v/total_roles:.1%}' for k, v in role_counts.items()})
    return pyg_data, all_scored


pyg_dataset, all_scored_docs = build_dataset(df, sample=500)

C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\2900861423.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(label_col, group_keys=False).apply(
Building graphs:   0%|          | 0/500 [00:00<?, ?it/s]C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   0%|          | 1/500 [00:04<40:23,  4.86s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   0%| 

  Search failed for "TRUMP TAPS Anti-Iran Deal Congressman To Head CIA…The Left G": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=TRUMP+TAPS+Anti-Iran+Deal+Congressman+To+Head+CIA%E2%80%A6The+Left+Goes+Ballistic%21)', 'https://www.bing.com/search?q=TRUMP+TAPS+Anti-Iran+Deal+Congressman+To+Head+CIA%E2%80%A6The+Left+Goes+Ballistic%21')


C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  53%|█████▎    | 267/500 [19:23<38:40,  9.96s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  54%|█████▎    | 268/500 [19:31<36:37,  9.47s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  54%|█████▍    | 269/500 [19:39<34:52,  9.06s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_7968\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Buildi


Built 493 | Augmented: 105 | Fallback: 388
Role distribution: {'claim': '75.2%', 'evidence': '0.1%', 'analysis': '9.4%', 'background': '15.3%'}


## Step 8b: Batch Build with Evidence Retry

Two new capabilities replace the monolithic `build_dataset` + single training pass:

### `build_batch`
Builds graphs for one slice of the dataframe. Unlike `build_dataset`, it separates
*failure tracking* from the hot path: articles that return zero evidence are stored in a
`failed_items` list along with their **pre-augmentation graph state** (base graph,
sentence list, role list) so that `retry_failed_evidence` can patch them without
re-running the expensive NLI role-classification pass.

### `retry_failed_evidence`
Runs up to three progressively broader fallback search strategies on failed articles:

1. **Truncated headline** — first six words (removes rare proper nouns that confuse DDG)
2. **NER keyword query** — spaCy extracts named entities + noun chunks → compact query
3. **Lead-sentence query** — first sentence of the article body

Each strategy is tried in order; as soon as one produces documents the graph is
augmented in-place and the item is removed from the failure list. Articles that still
fail after all retries are included as base-graph-only (no cross-source edges) so the
article is not silently dropped from the dataset.


In [10]:

from dataclasses import dataclass
from tqdm import tqdm
import copy


@dataclass
class FailedItem:
    """Holds the pre-augmentation state of an article that returned zero evidence,
    so retry_failed_evidence can patch it without re-running role classification."""
    df_idx:        int
    title:         str
    text:          str
    label:         int
    G_base:        object     # nx.Graph before augmentation
    article_sents: list
    article_roles: list


def build_batch(
    batch_df,
    text_col:  str  = 'text',
    title_col: str  = 'title',
    label_col: str  = 'label_binary',
    use_nli:   bool = True,
):
    """
    Build graphs for one batch of articles.

    Returns
    -------
    pyg_data     : list[Data]        — PyG graphs for augmented articles
    all_scored   : list[list]        — scored doc lists aligned with pyg_data
    failed_items : list[FailedItem]  — articles with zero evidence (can be retried)
    role_counts  : dict              — role distribution diagnostics
    """
    pyg_data, all_scored, failed_items = [], [], []
    n_aug = 0
    role_counts = {r: 0 for r in ROLE_NAMES}

    for df_idx, row in tqdm(batch_df.iterrows(), total=len(batch_df),
                            desc='Building batch', leave=False):
        text  = row[text_col]
        title = row.get(title_col, '')
        label = int(row[label_col])

        G_base, article_sents, article_roles = build_article_graph(text, use_nli=use_nli)
        if not article_sents:
            continue  # empty article — skip entirely

        for r in article_roles:
            role_counts[ROLE_NAMES[r]] += 1

        scored_docs  = []
        got_evidence = False
        try:
            docs = collect_evidence_by_headline(title, k=10)
            if docs:
                G_aug, scored_docs = augment_with_cross_source(
                    G_base, article_sents, article_roles, docs, use_nli=use_nli
                )
                pyg_data.append(graph_to_pyg(G_aug, label=label))
                all_scored.append(scored_docs)
                n_aug += 1
                got_evidence = True
        except Exception as e:
            print(f'  [build_batch] augmentation error idx={df_idx}: {e}')

        if not got_evidence:
            # Preserve pre-augmentation state for retry
            failed_items.append(FailedItem(
                df_idx=df_idx, title=title, text=text, label=label,
                G_base=copy.deepcopy(G_base),
                article_sents=article_sents,
                article_roles=article_roles,
            ))

    total_roles = max(sum(role_counts.values()), 1)
    print(f'  Built {len(pyg_data)} | Augmented: {n_aug} | Failed: {len(failed_items)}')
    print('  Role dist:', {k: f'{v/total_roles:.1%}' for k, v in role_counts.items()})
    return pyg_data, all_scored, failed_items, role_counts


# ─────────────────────────────────────────────────────────────────────────────

def _fallback_queries(item: FailedItem) -> list:
    """
    Return up to three progressively broader fallback queries for a failed article,
    ordered from most specific to most generic.
    """
    queries = []

    # Strategy 1: truncate headline to first 6 words
    words = clean_headline(item.title).split()
    if len(words) > 3:
        queries.append(' '.join(words[:6]))

    # Strategy 2: spaCy NER — named entities + top noun chunks from title + lead text
    doc  = nlp(item.title + '. ' + item.text[:300])
    ents = [e.text for e in doc.ents
            if e.label_ in ('PERSON', 'ORG', 'GPE', 'EVENT', 'NORP', 'FAC', 'LOC')]
    chunks = [c.root.text for c in doc.noun_chunks if len(c.root.text) > 3]
    kw = list(dict.fromkeys(ents + chunks))[:5]   # deduplicate, preserve order
    if kw:
        queries.append(' '.join(kw))

    # Strategy 3: first substantive sentence of the article body
    sents = [s.strip() for s in item.text.split('.') if len(s.strip()) > 20]
    if sents:
        queries.append(sents[0][:120])

    return queries


def retry_failed_evidence(
    failed_items: list,
    use_nli: bool = True,
):
    """
    Attempt to retrieve evidence for articles that returned zero results during
    build_batch, using up to three fallback query strategies per article.

    Articles that fail all strategies are included as base-graph-only (no cross-source
    edges) so no articles are silently dropped from the dataset.

    Returns
    -------
    new_pyg      : list[Data]        — graphs for all previously-failed articles
    new_scored   : list[list]        — aligned scored doc lists
    still_failed : list[FailedItem]  — articles that failed all strategies (base-only)
    """
    new_pyg, new_scored, still_failed = [], [], []

    for item in tqdm(failed_items, desc='Retrying evidence', leave=False):
        recovered = False
        for attempt, query in enumerate(_fallback_queries(item), start=1):
            try:
                docs = ddg_search(query, num_results=10)
                if not docs:
                    continue
                G_retry = copy.deepcopy(item.G_base)
                G_aug, scored_docs = augment_with_cross_source(
                    G_retry, item.article_sents, item.article_roles,
                    docs, use_nli=use_nli
                )
                if scored_docs:   # evidence nodes were actually added
                    new_pyg.append(graph_to_pyg(G_aug, label=item.label))
                    new_scored.append(scored_docs)
                    print(f'  [retry] idx={item.df_idx} recovered via '
                          f'strategy {attempt} ("{query[:50]}")')
                    recovered = True
                    break
            except Exception as e:
                print(f'  [retry] strategy {attempt} failed for idx={item.df_idx}: {e}')

        if not recovered:
            # Include base-graph (no cross-source edges) so article stays in dataset
            new_pyg.append(graph_to_pyg(item.G_base, label=item.label))
            new_scored.append([])
            still_failed.append(item)

    n_recovered = len(failed_items) - len(still_failed)
    print(f'[retry] Recovered: {n_recovered} | Still failed (base-only): {len(still_failed)}')
    return new_pyg, new_scored, still_failed


## Step 8c: Warm-Start Fine-Tuning

`finetune_gat` differs from `train_gat` in three ways:

1. **No re-initialisation** — accepts an existing model and continues from its
   current weights rather than constructing a new `FakeNewsGAT()`.
2. **Lower learning rate** — defaults to `1e-4` (vs `5e-4` for cold start) to
   avoid catastrophic forgetting of patterns learned in earlier batches.
3. **Cosine-annealing restart** — the scheduler's `T_max` is set to the
   fine-tuning epoch count, giving a fresh cosine curve per batch.


In [11]:

def finetune_gat(
    model,
    train_data,
    val_data,
    epochs:     int   = 30,
    lr:         float = 1e-4,
    patience:   int   = 8,
    batch_size: int   = 16,
):
    """
    Fine-tune an existing FakeNewsGAT on new data (warm start).

    Returns the updated model and a loss dict {'train_loss': ..., 'val_loss': ...}.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.BCEWithLogitsLoss()
    t_loader  = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    v_loader  = DataLoader(val_data,   batch_size=batch_size)

    best_val, no_improve, best_state = float('inf'), 0, None
    avg_t = avg_v = 0.0

    for epoch in range(epochs):
        model.train()
        t_loss = 0.0
        for b in t_loader:
            optimizer.zero_grad()
            out  = model(b.x, b.edge_index, b.batch).squeeze()
            loss = criterion(out, b.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        scheduler.step()

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for b in v_loader:
                v_loss += criterion(
                    model(b.x, b.edge_index, b.batch).squeeze(),
                    b.y.squeeze()
                ).item()

        avg_t = t_loss / max(len(t_loader), 1)
        avg_v = v_loss / max(len(v_loader), 1)

        if avg_v < best_val:
            best_val, no_improve = avg_v, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model, {'train_loss': avg_t, 'val_loss': best_val}


## Step 9b: Batched Training Loop

Replaces the monolithic Step 9 with an iterative loop that interleaves graph
construction and model training:

```
for each batch i:
    1. build_batch()              ← graph construction for this slice
    2. retry_failed_evidence()    ← recover articles with 0 search results
    3. split batch into train/val (for batch-local early stopping)
    4. if i == 0:  train_gat()     (cold start, more epochs)
       else:       finetune_gat()  (warm start, lower LR)
    5. evaluate on fixed global val pool → log per-batch metrics
    6. update credibility DB with this batch's predictions
    7. save checkpoint
```

### Why this helps

- **Faster feedback** — accuracy numbers appear after the first batch, not after
  the entire dataset is built.
- **Compounding DB** — the source credibility database grows richer each batch, so
  later batches score retrieved documents with more accumulated signal.
- **Visualised learning curve** — `batch_history` (accuracy / F1 / AUC per batch on a
  fixed global val pool) makes convergence visible.

### Parameters to tune

| Parameter | Default | Notes |
|-----------|---------|-------|
| `batch_size` | 50 | Articles per batch; smaller = more DB updates, noisier gradients |
| `val_pool_size` | 60 | Fixed global val set; set aside before any batching |
| `cold_epochs` | 80 | Epochs for batch 0 (cold start) |
| `warm_epochs` | 30 | Epochs per subsequent batch (warm start) |
| `val_split` | 0.2 | Fraction of each batch held out for batch-local early stopping |
| `retry` | True | Run `retry_failed_evidence` on each batch |

### Resuming after a kernel restart

```python
model = load_model('fake_news_gat_v4_batch003.pt')
# slice work_df from row batch_size*3 onward and pass to a new batched_train_loop call,
# or loop manually over the remaining batches using finetune_gat().
```


In [12]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split as _tts


def batched_train_loop(
    df,
    batch_size:    int   = 50,
    val_pool_size: int   = 60,
    cold_epochs:   int   = 80,
    warm_epochs:   int   = 30,
    val_split:     float = 0.2,
    retry:         bool  = True,
    use_nli:       bool  = True,
    save_prefix:   str   = 'fake_news_gat_v4',
    text_col:      str   = 'text',
    title_col:     str   = 'title',
    label_col:     str   = 'label_binary',
):
    """
    Iterative build-and-train loop with per-batch metric tracking.

    Returns the final model and batch_history (list of per-batch metric dicts).
    """
    # ── 1. Reserve a fixed global validation pool ─────────────────────────────
    pool_frac = val_pool_size / len(df)
    pool_df, work_df = _tts(df, test_size=(1 - pool_frac),
                            stratify=df[label_col], random_state=42)
    pool_df = pool_df.iloc[:val_pool_size].reset_index(drop=True)
    work_df = work_df.reset_index(drop=True)

    n_batches = (len(work_df) + batch_size - 1) // batch_size
    print(f'Global val pool : {len(pool_df)} articles')
    print(f'Working set     : {len(work_df)} articles -> ~{n_batches} batches of {batch_size}')

    # Build val-pool graphs once upfront (they never change)
    print('\n[init] Building global validation pool...')
    val_pool_pyg, _, val_pool_fails, _ = build_batch(
        pool_df, text_col, title_col, label_col, use_nli
    )
    if retry and val_pool_fails:
        extra, _, _ = retry_failed_evidence(val_pool_fails, use_nli)
        val_pool_pyg.extend(extra)
    print(f'[init] Val pool ready: {len(val_pool_pyg)} graphs\n')

    # ── 2. Batch loop ─────────────────────────────────────────────────────────
    model         = None
    batch_history = []
    all_pyg       = []
    all_scored    = []

    for b_idx in range(n_batches):
        start    = b_idx * batch_size
        end      = min(start + batch_size, len(work_df))
        batch_df = work_df.iloc[start:end]

        print('=' * 60)
        print(f'BATCH {b_idx+1}/{n_batches}  (rows {start}-{end-1}, n={len(batch_df)})')
        print('=' * 60)

        # 2a. Build graphs for this batch
        b_pyg, b_scored, b_failed, _ = build_batch(
            batch_df, text_col, title_col, label_col, use_nli
        )

        # 2b. Retry failed evidence
        if retry and b_failed:
            print(f'  Retrying {len(b_failed)} failed articles...')
            r_pyg, r_scored, _ = retry_failed_evidence(b_failed, use_nli)
            b_pyg.extend(r_pyg)
            b_scored.extend(r_scored)

        all_pyg.extend(b_pyg)
        all_scored.extend(b_scored)

        if len(b_pyg) < 4:
            print(f'  Batch too small ({len(b_pyg)} graphs) — skipping training step.')
            continue

        # 2c. Batch-local train / val split (for early stopping only)
        n_val   = max(1, int(len(b_pyg) * val_split))
        b_val   = b_pyg[:n_val]
        b_train = b_pyg[n_val:]
        if not b_train:
            b_train = b_val   # edge case: tiny batch

        # 2d. Train or fine-tune
        if model is None:
            print(f'  Cold start: {cold_epochs} epochs...')
            model = train_gat(b_train, b_val, epochs=cold_epochs, patience=15)
        else:
            print(f'  Warm start: {warm_epochs} epochs (lr=1e-4)...')
            model, ft = finetune_gat(model, b_train, b_val,
                                     epochs=warm_epochs, patience=8)
            print(f'  Fine-tune: train_loss={ft["train_loss"]:.4f}  '
                  f'val_loss={ft["val_loss"]:.4f}')

        # 2e. Evaluate on the fixed global val pool
        if val_pool_pyg:
            metrics, _ = evaluate_model(model, val_pool_pyg)
            print(f'  [global val] acc={metrics["accuracy"]:.3f}  '
                  f'f1={metrics["f1"]:.3f}  auc={metrics["auc"]:.3f}')
        else:
            metrics = {'accuracy': 0.0, 'f1': 0.0, 'auc': 0.0}

        batch_history.append({
            'batch':        b_idx + 1,
            'n_graphs':     len(b_pyg),
            'total_graphs': len(all_pyg),
            **metrics,
        })

        # 2f. Update credibility DB with this batch's predictions
        n_db = 0
        model.eval()
        with torch.no_grad():
            for graph, scored in zip(b_pyg, b_scored):
                if not scored:
                    continue
                from torch_geometric.loader import DataLoader as _DL
                for btch in _DL([graph], batch_size=1):
                    prob = torch.sigmoid(
                        model(btch.x, btch.edge_index, btch.batch).squeeze()
                    ).item()
                    conf = abs(prob - 0.5) * 2.0
                    n_db += bulk_update_from_prediction(
                        scored, prob, conf, confidence_threshold=0.1
                    )
        print(f'  Credibility DB: {n_db} entries updated.')

        # 2g. Save checkpoint
        ckpt = f'{save_prefix}_batch{b_idx+1:03d}.pt'
        save_model(model, ckpt)

    # ── 3. Plot learning curve ────────────────────────────────────────────────
    if batch_history:
        _plot_batch_history(batch_history, save_prefix)

    print('\n[done] Batched training complete.')
    return model, batch_history


def _plot_batch_history(history, prefix):
    """Save a 2-panel learning-curve figure: metrics + dataset growth."""
    batches = [h['batch']        for h in history]
    accs    = [h['accuracy']     for h in history]
    f1s     = [h['f1']           for h in history]
    aucs    = [h['auc']          for h in history]
    totals  = [h['total_graphs'] for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(batches, accs, marker='o', label='Accuracy')
    ax1.plot(batches, f1s,  marker='s', label='F1')
    ax1.plot(batches, aucs, marker='^', label='AUC-ROC')
    ax1.set_xlabel('Batch')
    ax1.set_ylabel('Score (global val pool)')
    ax1.set_title('Model Performance per Batch')
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.bar(batches, totals, color='steelblue', alpha=0.7)
    ax2.set_xlabel('Batch')
    ax2.set_ylabel('Cumulative graphs built')
    ax2.set_title('Dataset Growth per Batch')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    out = f'{prefix}_learning_curve.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'[plot] Learning curve saved -> {out}')


# ── Entry point ───────────────────────────────────────────────────────────────
# Recommended config for a 500-article sample:
#
#   final_model, history = batched_train_loop(
#       df,
#       batch_size    = 50,
#       val_pool_size = 60,
#       cold_epochs   = 80,
#       warm_epochs   = 30,
#       retry         = True,
#   )


## Step 9: Train, Evaluate, and Update Credibility DB

We do a 70 / 15 / 15 train/val/test split, train the GAT with early stopping, evaluate
on the held-out test set, and then run inference on the full dataset to update the source
credibility database with the model's predictions.

In [13]:
indices             = list(range(len(pyg_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
test_data  = [pyg_dataset[i] for i in test_idx]
print(f'Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test')

model = train_gat(train_data, val_data, epochs=150, patience=20)

metrics, test_probs = evaluate_model(model, test_data)
print(f'\nTest Results:')
print(f'  Accuracy : {metrics["accuracy"]:.3f}')
print(f'  F1       : {metrics["f1"]:.3f}')
print(f'  AUC-ROC  : {metrics["auc"]:.3f}')

# Update credibility DB
model.eval()
n_db = 0
with torch.no_grad():
    for batch, scored in zip(DataLoader(pyg_dataset, batch_size=1), all_scored_docs):
        if not scored:
            continue
        prob = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch).squeeze()).item()
        conf = abs(prob - 0.5) * 2
        n_db += bulk_update_from_prediction(scored, prob, conf, confidence_threshold=0.1)

print(f'\nCredibility DB: {n_db} entries updated.')
save_model(model, 'fake_news_gat_v3.pt')

Split: 345 train | 74 val | 74 test


c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params

Epoch  10  train=0.5465  val=0.6270  lr=4.95e-04
Epoch  20  train=0.5312  val=0.6486  lr=4.78e-04
Early stopping at epoch 25

Test Results:
  Accuracy : 0.757
  F1       : 0.757
  AUC-ROC  : 0.832

Credibility DB: 676 entries updated.
Saved to fake_news_gat_v3.pt


In [14]:
with get_db() as conn:
    rows = conn.execute(
        'SELECT domain, alpha, beta, model_updates, user_updates '
        'FROM sources ORDER BY model_updates + user_updates DESC LIMIT 25'
    ).fetchall()

print(f'{"Domain":<40} {"Score":>6} {"Signals":>8} {"Model":>7} {"User":>6}')
print('-' * 70)
for domain, alpha, beta, m, u in rows:
    score = alpha / (alpha + beta)
    n     = int(alpha + beta - 4)
    print(f'{domain:<40} {score:>6.3f} {n:>8} {m:>7} {u:>6}')

Domain                                    Score  Signals   Model   User
----------------------------------------------------------------------
en.wikipedia.org                          0.615        6     131      0
en.m.wikipedia.org                        0.581        5     105      0
britannica.com                            0.549        3      82      0
apnews.com                                0.557        3      64      0
youtube.com                               0.494        2      63      0
whitehouse.gov                            0.576        3      62      0
merriam-webster.com                       0.415        2      59      0
dictionary.cambridge.org                  0.474        2      44      0
usatoday.com                              0.549        2      43      0
nytimes.com                               0.544        1      39      0
nbcnews.com                               0.543        1      38      0
time.com                                  0.543        1      36 

## Architecture Improvement Proposals

These are concrete changes that could improve the model, in rough order of
expected impact.

---

### 1. Replace `GATConv` with `GATv2Conv` *(high impact, trivial change)*

The original GAT attention mechanism has a theoretical limitation: it computes attention
weights before combining the query and key representations, making it equivalent to a
static (input-independent) attention in certain graph structures. GATv2 fixes this by
applying the non-linearity *after* concatenating the node representations, making
attention genuinely dynamic.

```python
# Change in imports:
from torch_geometric.nn import GATv2Conv  # replaces GATConv

# Change in __init__: replace GATConv(...) → GATv2Conv(...)
# The API is identical; no other changes needed.
self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=n_heads, concat=True, dropout=dropout)
```

---

### 2. Feed edge features into attention *(medium impact)*

The graph carries rich edge attributes (edge type, NLI confidence, similarity, cross-source flag)
but `GATConv` / `GATv2Conv` can only use them if you pass `edge_dim`. Currently they are
computed but ignored during message passing.

```python
EDGE_DIM = 7  # matches the edge_attr dimension in graph_to_pyg()

self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=n_heads,
                       concat=True, dropout=dropout, edge_dim=EDGE_DIM)
# ... same for conv2, conv3, conv4

# In forward():
x1 = self.bn1(F.relu(self.lin1(self.conv1(x, edge_index, edge_attr=edge_attr)))) + x
```

You would also need to add `edge_attr` as a parameter to `forward()` and pass `b.edge_attr`
in the training loop.

---

### 3. Use a richer sentence encoder *(medium impact)*

`all-MiniLM-L6-v2` is fast but relatively weak for semantic nuance. Upgrading to
`all-mpnet-base-v2` (same API, 420 MB vs 80 MB) consistently gives +2–4 points on
STS benchmarks and would produce better embeddings for both the graph edges and
NLI role classification.

```python
ENCODER = SentenceTransformer('all-mpnet-base-v2')
```

Alternatively, `BAAI/bge-small-en-v1.5` is only marginally larger than MiniLM but
significantly stronger.

---

### 4. Replace column-max normalisation with Z-score standardisation *(low-medium impact)*

The current normalisation divides each feature column by its maximum value. This is
sensitive to outliers (one very large value collapses all others toward 0) and doesn't
centre the features, which can slow down learning.

```python
from sklearn.preprocessing import StandardScaler

# In build_dataset, after collecting all pyg_data:
all_x = torch.cat([d.x for d in pyg_data], dim=0).numpy()
scaler = StandardScaler().fit(all_x)

for d in pyg_data:
    d.x = torch.tensor(scaler.transform(d.x.numpy()), dtype=torch.float)
```

Save the scaler alongside the model weights so you can normalise at inference time.

---

### 5. Add a heterogeneous graph formulation *(high impact, more work)*

Right now, `input` and `evidence` nodes are structurally identical in the model — only
feature 6 (`is_evidence`) distinguishes them. PyG's `HeteroData` lets you define
separate embedding spaces and message-passing weights for different node/edge types,
which is a much more principled way to handle this.

```python
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

data = HeteroData()
data['input'].x    = input_features
data['evidence'].x = evidence_features
data['input',  'similar_to', 'input'].edge_index    = intra_edges
data['input',  'supported_by', 'evidence'].edge_index = cross_edges
data['input',  'contradicted_by', 'evidence'].edge_index = contra_edges
```

This is the most architecturally significant change but also the largest refactor.

---

### 6. Add a role-prediction auxiliary loss *(low-medium impact)*

If you have any ground-truth role labels (or can create a small labelled sample with
`classify_sentence_roles` as a noisy teacher), you can add a node-level auxiliary loss
that forces the model to learn role-aware representations. Multi-task learning often
improves the primary task even when the auxiliary task is noisy.

```python
# In forward(), add a branch from the node embeddings before pooling:
role_logits = self.role_head(xjk)  # shape (n_nodes, 4)

# Training loss:
loss = bce_loss(graph_logit, label) + 0.1 * ce_loss(role_logits, node_role_labels)
```